# 03 — Cross-Analysis of Automatic Metrics and Human Evaluation — Revised Version

This notebook builds the **final statistical dataset** for analyzing the association between human evaluation and automatic metrics.

## Expected experiment structure

- **5 models**
- **3 techniques** per model:
  - zero-shot
  - one-shot
  - few-shot
- **4 main metric-file families**:
  - Manhattan
  - METEOR
  - BERTScore
  - NLI

Therefore:

\[
5 \times 3 \times 4 = 60\ CSV\ files
\]

Each file for a `model × technique` condition should ideally contain:

- 259 cases;
- 10 executions per case;
- 2,590 unique `generation_id` values.

## Inputs

```text
avaliacao_humana_consolidada.json

metricas/
    metricas_manhattan_....csv
    metricas_meteor_....csv
    metricas_bertscore_....csv
    metricas_nli_....csv
    ...
```

All CSV files may be stored together in the same folder.

## How the notebook avoids mixing files

The notebook **does not use row order** and does not depend on the file name to perform the cross-analysis.

For each CSV file, it:

1. identifies the metric family from its columns;
2. reads `modelo` and `tecnica` from the file content itself;
3. validates that the file contains only one model and one technique;
4. validates `generation_id`;
5. builds a `family × model × technique` inventory;
6. detects duplicate combinations;
7. detects missing combinations;
8. compares the `generation_id` sets across the four families for the same condition;
9. filters only the `generation_id` values that were evaluated by humans;
10. joins using the exact `generation_id` key.

Thus, the row for execution 7 will never be confused with executions 1, 2, 3, etc.

## Main metrics used in the correlation analysis

By default:

- Manhattan → `distancia_manhattan`
- METEOR → `meteor_score`
- BERTScore → `bertscore_f1`
- NLI → `nli_score`

BERTScore and NLI components can be enabled as secondary analyses.

## Statistical analyses

The notebook performs:

- metric coverage analysis;
- descriptive statistics;
- Shapiro–Wilk;
- histograms;
- Q-Q plots;
- IQR-based outlier analysis;
- linearity diagnostics;
- Pearson correlation;
- Spearman correlation;
- pair-specific method recommendation;
- 95% bootstrap confidence intervals;
- Holm correction;
- available-pairs analysis;
- complete-case sensitivity analysis.


In [ ]:

from pathlib import Path
from collections import Counter
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats


## 1. Configuration


In [ ]:

# ============================================================
# FILES AND FOLDERS
# ============================================================

EVALUATION_FILE = Path("avaliacao_humana_consolidada.json")
METRICS_FOLDER = Path("metricas")

OUTPUT_FOLDER = Path("resultados_correlacao")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# EXPECTED EXPERIMENTAL DESIGN
# ============================================================

EXPECTED_MODEL_COUNT = 5
EXPECTED_TECHNIQUE_COUNT = 3

EXPECTED_TECHNIQUES = {
    "zero-shot",
    "one-shot",
    "few-shot"
}

REQUIRED_FAMILIES = {
    "manhattan",
    "meteor",
    "bertscore",
    "nli"
}

EXPECTED_TOTAL_CASES_PER_FILE = 259
EXPECTED_EXECUTIONS_PER_CASE = 10

EXPECTED_TOTAL_ROWS_PER_FILE = (
    EXPECTED_TOTAL_CASES_PER_FILE
    * EXPECTED_EXECUTIONS_PER_CASE
)

EXPECTED_TOTAL_FILES = (
    EXPECTED_MODEL_COUNT
    * EXPECTED_TECHNIQUE_COUNT
    * len(REQUIRED_FAMILIES)
)


# ============================================================
# VALIDATION CONTROLS
# ============================================================

# True = stop execution if any of the 60 files is missing.
# During partial tests, this can be changed to False.
REQUIRE_COMPLETE_60_FILE_SET = True

# True = require 2,590 rows and unique IDs per file.
REQUIRE_2590_RECORDS_PER_FILE = True

# True = require that, for the same model+technique combination,
# Manhattan, METEOR, BERTScore, and NLI contain exactly
# the same set of generation_id values.
REQUIRE_SAME_IDS_ACROSS_FAMILIES = True


# ============================================================
# HUMAN VARIABLES
# ============================================================

# The correlation uses only the human final score.
# Structure, semantics, and details are used only to compose
# the final score in Notebook 2 and are not included in the statistical analyses.
HUMAN_VARIABLES = [
    "nota_final"
]


# ============================================================
# MAIN METRICS
# ============================================================

# These four are the main metrics in the analysis.
MAIN_METRICS = [
    "manhattan",
    "meteor",
    "bertscore_f1",
    "nli_score"
]

# If True, auxiliary components are also correlated,
# increasing the number of statistical tests.
ANALYZE_SECONDARY_COMPONENTS = False


# ============================================================
# STATISTICAL PARAMETERS
# ============================================================

ALPHA = 0.05

# 5000 is suitable for the final analysis.
# It can be increased to 10000 if desired.
N_BOOTSTRAP = 5000

# Bootstrap specifically for directly comparing correlations
# between metrics. Since there are only 6 comparisons among
# the 4 main metrics, 10,000 resamples are used.
N_BOOTSTRAP_COMPARISON = 10000

SEED_BOOTSTRAP = 20260827

# If the quadratic model improves R² by more than 0.05
# over the linear model, possible nonlinearity is flagged.
NONLINEAR_DELTA_R2_THRESHOLD = 0.05

GENERATE_PLOTS_FOR_ALL_CRITERIA = False


## 2. Family and Column Configuration


In [ ]:

# ============================================================
# HOW TO IDENTIFY THE FILE FAMILY
# ============================================================
#
# The family is identified by the CSV columns, not by the
# file name.
# ============================================================

FAMILY_CONFIG = {
    "manhattan": {
        "assinatura": {
            "distancia_manhattan"
        }
    },

    "meteor": {
        "assinatura": {
            "meteor_score"
        }
    },

    "bertscore": {
        "assinatura": {
            "bertscore_precision",
            "bertscore_recall",
            "bertscore_f1"
        }
    },

    "nli": {
        "assinatura": {
            "nli_score",
            "entailment_mean",
            "neutral_mean",
            "contradiction_mean"
        }
    }
}


# ============================================================
# NUMERIC METRICS THAT CAN BE EXTRACTED
# ============================================================
#
# principal=True:
#   included in the default analysis.
#
# principal=False:
#   included only if ANALYZE_SECONDARY_COMPONENTS=True.
#
# orientacao:
#   +1 = higher is better
#   -1 = lower is better
# ============================================================

METRICS_CONFIG = {
    "manhattan": {
        "familia": "manhattan",
        "coluna": "distancia_manhattan",
        "orientacao": -1,
        "principal": True
    },

    "meteor": {
        "familia": "meteor",
        "coluna": "meteor_score",
        "orientacao": +1,
        "principal": True
    },

    "bertscore_f1": {
        "familia": "bertscore",
        "coluna": "bertscore_f1",
        "orientacao": +1,
        "principal": True
    },

    "nli_score": {
        "familia": "nli",
        "coluna": "nli_score",
        "orientacao": +1,
        "principal": True
    },

    # --------------------------------------------------------
    # SECONDARY COMPONENTS
    # --------------------------------------------------------

    "bertscore_precision": {
        "familia": "bertscore",
        "coluna": "bertscore_precision",
        "orientacao": +1,
        "principal": False
    },

    "bertscore_recall": {
        "familia": "bertscore",
        "coluna": "bertscore_recall",
        "orientacao": +1,
        "principal": False
    },

    "nli_entailment": {
        "familia": "nli",
        "coluna": "entailment_mean",
        "orientacao": +1,
        "principal": False
    },

    "nli_neutral": {
        "familia": "nli",
        "coluna": "neutral_mean",
        "orientacao": -1,
        "principal": False
    },

    "nli_contradiction": {
        "familia": "nli",
        "coluna": "contradiction_mean",
        "orientacao": -1,
        "principal": False
    },

    "nli_contradiction_rate": {
        "familia": "nli",
        "coluna": "contradiction_rate",
        "orientacao": -1,
        "principal": False
    },

    "nli_coverage": {
        "familia": "nli",
        "coluna": "coverage",
        "orientacao": +1,
        "principal": False
    }
}


### Important NLI Correction

In this version:

```text
nli_score
```

is treated as the **main NLI metric**.

It is not confused with:

```text
entailment_mean
```

Both remain available as different variables.

Likewise, for BERTScore, the main variable is:

```text
bertscore_f1
```

while `precision`/`recall` remain secondary analyses.


## 3. Helper Functions


In [ ]:

def load_json(path):
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path.resolve()}"
        )

    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def read_csv_robustly(path):
    attempts = [
        {"encoding": "utf-8-sig"},
        {"encoding": "utf-8"},
        {"encoding": "latin1"}
    ]

    last_error = None

    for options in attempts:
        try:
            df = pd.read_csv(
                path,
                **options
            )

            if len(df.columns) == 1:
                df2 = pd.read_csv(
                    path,
                    sep=";",
                    **options
                )

                if len(df2.columns) > 1:
                    df = df2

            return df

        except Exception as error:
            last_error = error

    raise RuntimeError(
        f"Could not read {path}: {last_error}"
    )


def normalize_column_name(name):
    name = str(name).strip().lower()

    replacements = {
        "á": "a",
        "à": "a",
        "ã": "a",
        "â": "a",
        "é": "e",
        "ê": "e",
        "í": "i",
        "ó": "o",
        "ô": "o",
        "õ": "o",
        "ú": "u",
        "ç": "c"
    }

    for source, target in replacements.items():
        name = name.replace(
            source,
            target
        )

    name = re.sub(
        r"[^a-z0-9]+",
        "_",
        name
    )

    return name.strip("_")


def standardize_columns(df):
    df = df.copy()

    df.columns = [
        normalize_column_name(column)
        for column in df.columns
    ]

    aliases = {
        "modelo": "model",
        "tecnica": "technique",
        "execucao": "execution"
    }

    df = df.rename(
        columns={
            old_name: new_name
            for old_name, new_name in aliases.items()
            if old_name in df.columns
        }
    )

    return df


def detect_family(df):
    columns = set(
        df.columns
    )

    found_items = []

    for family, config in FAMILY_CONFIG.items():
        if config["assinatura"].issubset(columns):
            found_items.append(family)

    if len(found_items) == 0:
        return None

    if len(found_items) > 1:
        raise ValueError(
            f"The CSV corresponds to more than one family: {found_items}"
        )

    return found_items[0]


def require_single_value(df, column, file):
    if column not in df.columns:
        raise ValueError(
            f"{file}: required column missing: {column}"
        )

    values = (
        df[column]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    if len(values) != 1:
        raise ValueError(
            f"{file}: expected exactly one value in '{column}', "
            f"but found {len(values)}: {values[:10]}"
        )

    return values[0]


## 4. Load the Consolidated Human Evaluation


In [ ]:

evaluation_data = load_json(
    EVALUATION_FILE
)

if "resultados" not in evaluation_data:
    raise ValueError(
        "The consolidated JSON does not contain the 'resultados' key."
    )

human_df = pd.DataFrame(
    evaluation_data["resultados"]
)

required_columns = [
    "avaliacao_id",
    "case_id",
    "model",
    "technique",
    "execution",
    "generation_id",
    "estrutura",
    "semantica",
    "detalhes",
    "nota_final"
]

missing_items = [
    column
    for column in required_columns
    if column not in human_df.columns
]

if missing_items:
    raise ValueError(
        f"Missing columns in the human evaluation: {missing_items}"
    )

if human_df["generation_id"].isna().any():
    raise ValueError(
        "There are evaluations without generation_id."
    )

if human_df["generation_id"].duplicated().any():
    raise ValueError(
        "There are duplicate generation_id values in the human evaluation."
    )

human_df["execution"] = pd.to_numeric(
    human_df["execution"],
    errors="raise"
).astype(int)

HUMAN_IDS = set(
    human_df["generation_id"]
)

print(
    f"Human evaluations: {len(human_df)}"
)

print(
    f"Unique generation IDs: {len(HUMAN_IDS)}"
)

display(
    human_df[
        [
            "avaliacao_id",
            "case_id",
            "model",
            "technique",
            "execution",
            "generation_id",
            "nota_final"
        ]
    ].head()
)


## 5. Locate All CSV Files


In [ ]:

metric_files = sorted(
    METRICS_FOLDER.rglob("*.csv")
)

if not metric_files:
    raise RuntimeError(
        f"No CSV file found in {METRICS_FOLDER.resolve()}"
    )

print(
    f"CSV files found: {len(metric_files)}"
)

print(
    f"Files expected by the current design: {EXPECTED_TOTAL_FILES}"
)


## 6. Build the File Inventory


In [ ]:

inventory = []
dataframes_by_key = {}
ids_by_key = {}

for path in metric_files:

    df = read_csv_robustly(
        path
    )

    df = standardize_columns(
        df
    )

    family = detect_family(
        df
    )

    if family is None:
        raise ValueError(
            f"Could not identify the family of {path.name}. "
            f"Columns found: {list(df.columns)}"
        )

    model = require_single_value(
        df,
        "model",
        path.name
    )

    technique = require_single_value(
        df,
        "technique",
        path.name
    )

    if "generation_id" not in df.columns:
        raise ValueError(
            f"{path.name}: generation_id missing."
        )

    if "case_id" not in df.columns:
        raise ValueError(
            f"{path.name}: case_id missing."
        )

    if "execution" not in df.columns:
        raise ValueError(
            f"{path.name}: execution/execucao missing."
        )

    df["execution"] = pd.to_numeric(
        df["execution"],
        errors="raise"
    ).astype(int)

    key = (
        family,
        model,
        technique
    )

    if key in dataframes_by_key:
        previous = dataframes_by_key[key]["arquivo"]

        raise ValueError(
            "Duplicate combination detected:\n"
            f"  family: {family}\n"
            f"  model: {model}\n"
            f"  technique: {technique}\n"
            f"  file 1: {previous}\n"
            f"  file 2: {path.name}"
        )

    n_rows = len(df)
    n_generation_ids = df["generation_id"].nunique()
    n_cases = df["case_id"].nunique()

    duplicate_generation_ids = int(
        df["generation_id"].duplicated().sum()
    )

    execution_counts = (
        df.groupby("case_id")["execution"]
        .nunique()
    )

    incomplete_execution_cases = int(
        (execution_counts != EXPECTED_EXECUTIONS_PER_CASE).sum()
    )

    ids = set(
        df["generation_id"]
    )

    inventory.append({
        "familia": family,
        "modelo": model,
        "tecnica": technique,
        "arquivo": path.name,
        "linhas": n_rows,
        "casos_unicos": n_cases,
        "generation_ids_unicos": n_generation_ids,
        "generation_ids_duplicados": duplicate_generation_ids,
        "casos_sem_10_execucoes": incomplete_execution_cases,
        "ids_humanos_encontrados_neste_arquivo": len(
            HUMAN_IDS & ids
        )
    })

    dataframes_by_key[key] = {
        "arquivo": path.name,
        "df": df
    }

    ids_by_key[key] = ids


inventory_df = pd.DataFrame(
    inventory
).sort_values(
    [
        "modelo",
        "tecnica",
        "familia"
    ]
).reset_index(drop=True)

display(inventory_df)


## 7. Validate the 5 Models × 3 Techniques × 4 Families Structure


In [ ]:

found_models = sorted(
    inventory_df["modelo"].unique()
)

found_techniques = set(
    inventory_df["tecnica"].unique()
)

found_families = set(
    inventory_df["familia"].unique()
)

print("Models found:")
for model in found_models:
    print(" -", model)

print(
    "\nTechniques found:",
    sorted(found_techniques)
)

print(
    "\nFamilies found:",
    sorted(found_families)
)

structure_problems = []

if len(found_models) != EXPECTED_MODEL_COUNT:
    structure_problems.append(
        f"Expected {EXPECTED_MODEL_COUNT} models, "
        f"found {len(found_models)}."
    )

if found_techniques != EXPECTED_TECHNIQUES:
    structure_problems.append(
        f"Expected techniques: {sorted(EXPECTED_TECHNIQUES)}; "
        f"found: {sorted(found_techniques)}."
    )

if not REQUIRED_FAMILIES.issubset(found_families):
    structure_problems.append(
        f"Missing required families: "
        f"{sorted(REQUIRED_FAMILIES - found_families)}"
    )


# ============================================================
# PRESENCE MATRIX
# ============================================================

expected_combinations = []

for model in found_models:
    for technique in sorted(EXPECTED_TECHNIQUES):
        for family in sorted(REQUIRED_FAMILIES):
            expected_combinations.append(
                (family, model, technique)
            )

found_keys = set(
    dataframes_by_key.keys()
)

missing_combinations = [
    key
    for key in expected_combinations
    if key not in found_keys
]

if missing_combinations:
    structure_problems.append(
        f"There are {len(missing_combinations)} combinations "
        "family × model × technique combinations are missing."
    )


if len(metric_files) != EXPECTED_TOTAL_FILES:
    structure_problems.append(
        f"Expected {EXPECTED_TOTAL_FILES} CSVs, "
        f"found {len(metric_files)}."
    )


if missing_combinations:
    print("\nMISSING COMBINATIONS:")

    for family, model, technique in missing_combinations:
        print(
            f" - {family} | {model} | {technique}"
        )


if structure_problems:
    print("\nPROBLEMS FOUND:")

    for problem in structure_problems:
        print(" -", problem)

    if REQUIRE_COMPLETE_60_FILE_SET:
        raise RuntimeError(
            "The complete file structure requirement was not met."
        )
else:
    print(
        "\nComplete structure validated: "
        f"{EXPECTED_TOTAL_FILES} files."
    )


## 8. Validate 2,590 Records and 10 Executions per Case


In [ ]:

content_problems = []

for _, row in inventory_df.iterrows():

    identification = (
        f"{row['familia']} | "
        f"{row['modelo']} | "
        f"{row['tecnica']}"
    )

    if row["linhas"] != EXPECTED_TOTAL_ROWS_PER_FILE:
        content_problems.append(
            f"{identification}: "
            f"{row['linhas']} rows; "
            f"expected {EXPECTED_TOTAL_ROWS_PER_FILE}."
        )

    if row["casos_unicos"] != EXPECTED_TOTAL_CASES_PER_FILE:
        content_problems.append(
            f"{identification}: "
            f"{row['casos_unicos']} cases; "
            f"expected {EXPECTED_TOTAL_CASES_PER_FILE}."
        )

    if row["generation_ids_unicos"] != EXPECTED_TOTAL_ROWS_PER_FILE:
        content_problems.append(
            f"{identification}: "
            f"{row['generation_ids_unicos']} unique generation_id values; "
            f"expected {EXPECTED_TOTAL_ROWS_PER_FILE}."
        )

    if row["generation_ids_duplicados"] != 0:
        content_problems.append(
            f"{identification}: "
            f"{row['generation_ids_duplicados']} duplicate generation_id values."
        )

    if row["casos_sem_10_execucoes"] != 0:
        content_problems.append(
            f"{identification}: "
            f"{row['casos_sem_10_execucoes']} cases without 10 executions."
        )


if content_problems:
    print("CONTENT PROBLEMS:")

    for problem in content_problems:
        print(" -", problem)

    if REQUIRE_2590_RECORDS_PER_FILE:
        raise RuntimeError(
            "There are incomplete or inconsistent files."
        )

else:
    print(
        "All files have the expected structure: "
        "259 cases × 10 executions = 2,590 records."
    )


## 9. Verify That the Four Families Contain the Same IDs Within Each Condition


In [ ]:

id_divergences = []

for model in found_models:
    for technique in sorted(EXPECTED_TECHNIQUES):

        condition_keys = [
            (
                family,
                model,
                technique
            )
            for family in sorted(REQUIRED_FAMILIES)
            if (
                family,
                model,
                technique
            ) in ids_by_key
        ]

        if len(condition_keys) < 2:
            continue

        reference_key = condition_keys[0]
        reference_ids = ids_by_key[
            reference_key
        ]

        for current_key in condition_keys[1:]:
            current_ids = ids_by_key[
                current_key
            ]

            missing_in_current = (
                reference_ids
                - current_ids
            )

            extras_in_current = (
                current_ids
                - reference_ids
            )

            if missing_in_current or extras_in_current:
                id_divergences.append({
                    "modelo": model,
                    "tecnica": technique,
                    "familia_referencia": reference_key[0],
                    "familia_comparada": current_key[0],
                    "faltando_na_comparada": len(missing_in_current),
                    "extras_na_comparada": len(extras_in_current)
                })


id_divergences_df = pd.DataFrame(
    id_divergences
)

if not id_divergences_df.empty:
    display(id_divergences_df)

    if REQUIRE_SAME_IDS_ACROSS_FAMILIES:
        raise RuntimeError(
            "The metric families do not contain the same set "
            "of generation_id values within some conditions."
        )
else:
    print(
        "Validation completed: within each model × technique condition, "
        "the four families contain the same set of generation_id values."
    )


## 10. Immediately Filter Only the 259 IDs Evaluated by Humans


In [ ]:

metrics_long = []
coverage_by_file = []

for key, content in dataframes_by_key.items():

    family, model, technique = key
    file = content["arquivo"]
    df = content["df"]

    filtered_df = df[
        df["generation_id"].isin(
            HUMAN_IDS
        )
    ].copy()

    coverage_by_file.append({
        "familia": family,
        "modelo": model,
        "tecnica": technique,
        "arquivo": file,
        "linhas_originais": len(df),
        "linhas_selecionadas_para_avaliacao_humana": len(filtered_df)
    })

    for metric_name, config in METRICS_CONFIG.items():

        if config["familia"] != family:
            continue

        if (
            not config["principal"]
            and not ANALYZE_SECONDARY_COMPONENTS
        ):
            continue

        value_column = config["coluna"]

        if value_column not in filtered_df.columns:
            raise ValueError(
                f"{file}: expected column '{value_column}' "
                f"was not found for metric {metric_name}."
            )

        temp = filtered_df[
            [
                "generation_id",
                "case_id",
                "model",
                "technique",
                "execution",
                value_column
            ]
        ].copy()

        temp = temp.rename(
            columns={
                value_column: "valor"
            }
        )

        temp["metrica"] = metric_name
        temp["familia"] = family
        temp["orientacao"] = config["orientacao"]
        temp["arquivo_origem"] = file

        temp["valor"] = pd.to_numeric(
            temp["valor"],
            errors="coerce"
        )

        metrics_long.append(
            temp
        )


file_coverage_df = pd.DataFrame(
    coverage_by_file
)

print(
    "The complete files were reduced to the records "
    "actually evaluated by humans."
)

display(
    file_coverage_df
)


### Why filter before concatenating?

With 60 files containing 2,590 rows each, the raw dataset may contain up to:

\[
60 \times 2590 = 155,400\ rows
\]

However, only 259 BDDs were evaluated by humans.

Therefore, each file is first filtered using the set of human-evaluated `generation_id` values. The selected records are concatenated afterward.


## 11. Consolidate the Selected Metrics


In [ ]:

if not metrics_long:
    raise RuntimeError(
        "No metric was extracted."
    )

metrics_long_df = pd.concat(
    metrics_long,
    ignore_index=True
)

# generation_id must appear at most once per metric
duplicates = (
    metrics_long_df
    .groupby(
        [
            "metrica",
            "generation_id"
        ]
    )
    .size()
    .reset_index(
        name="quantidade"
    )
)

duplicates = duplicates[
    duplicates["quantidade"] > 1
]

if not duplicates.empty:
    display(
        duplicates.head(50)
    )

    raise ValueError(
        "There are duplicate generation_id values within the same metric."
    )


detected_metrics = sorted(
    metrics_long_df["metrica"].unique()
)

print(
    "Metrics included in the analysis:"
)

for metric in detected_metrics:
    count = int(
        (
            metrics_long_df["metrica"]
            == metric
        ).sum()
    )

    print(
        f" - {metric}: {count} records"
    )


## 12. Redundant Validation of Case, Model, Technique, and Execution


In [ ]:

human_index = (
    human_df[
        [
            "generation_id",
            "case_id",
            "model",
            "technique",
            "execution"
        ]
    ]
    .copy()
    .rename(
        columns={
            "case_id": "case_id_humano",
            "model": "model_humano",
            "technique": "technique_humano",
            "execution": "execution_humano"
        }
    )
)

validation_df = metrics_long_df.merge(
    human_index,
    on="generation_id",
    how="inner"
)

inconsistencies = []

for metric_column, human_column in [
    ("case_id", "case_id_humano"),
    ("model", "model_humano"),
    ("technique", "technique_humano"),
    ("execution", "execution_humano")
]:

    left_values = validation_df[
        metric_column
    ]

    right_values = validation_df[
        human_column
    ]

    mask = (
        left_values.notna()
        & right_values.notna()
    )

    if metric_column == "execution":
        e = pd.to_numeric(
            left_values[mask],
            errors="coerce"
        )

        d = pd.to_numeric(
            right_values[mask],
            errors="coerce"
        )

    else:
        e = (
            left_values[mask]
            .astype(str)
            .str.strip()
        )

        d = (
            right_values[mask]
            .astype(str)
            .str.strip()
        )

    different = (
        e != d
    )

    if different.any():
        indices = different.index[
            different
        ]

        inconsistencies.append({
            "campo": metric_column,
            "quantidade": len(indices),
            "exemplos": validation_df.loc[
                indices,
                "generation_id"
            ].head(10).tolist()
        })


if inconsistencies:
    display(
        pd.DataFrame(
            inconsistencies
        )
    )

    raise RuntimeError(
        "The metric metadata do not match "
        "the human evaluation."
    )

print(
    "Redundant validation completed: "
    "case_id, model, technique, and execution are consistent."
)


## 13. Final Coverage by Metric


In [ ]:

coverage_records = []
missing_ids_by_metric = {}

for metric in detected_metrics:

    temp = metrics_long_df[
        metrics_long_df["metrica"]
        == metric
    ]

    metric_ids = set(
        temp["generation_id"]
    )

    found_ids = (
        HUMAN_IDS
        & metric_ids
    )

    missing_items = (
        HUMAN_IDS
        - metric_ids
    )

    missing_ids_by_metric[
        metric
    ] = sorted(
        missing_items
    )

    coverage_records.append({
        "metrica": metric,
        "avaliacoes_humanas": len(HUMAN_IDS),
        "encontrados": len(found_ids),
        "faltantes": len(missing_items),
        "cobertura_percentual": round(
            100 * len(found_ids)
            / len(HUMAN_IDS),
            2
        )
    })


coverage_df = pd.DataFrame(
    coverage_records
)

display(
    coverage_df
)


for metric, missing_items in missing_ids_by_metric.items():

    if not missing_items:
        continue

    print(
        f"\n{metric}: {len(missing_items)} missing human IDs"
    )

    for generation_id in missing_items[:20]:
        print(
            "  ",
            generation_id
        )

    if len(missing_items) > 20:
        print(
            f"  ... and {len(missing_items) - 20}"
        )


## 14. Create the Final Correlation Dataset


In [ ]:

metric_values_df = metrics_long_df[
    [
        "generation_id",
        "metrica",
        "valor"
    ]
].copy()

metrics_wide_df = (
    metric_values_df
    .pivot(
        index="generation_id",
        columns="metrica",
        values="valor"
    )
    .reset_index()
)

metrics_wide_df.columns.name = None

correlation_df = human_df.merge(
    metrics_wide_df,
    on="generation_id",
    how="left",
    validate="one_to_one"
)

if len(correlation_df) != len(human_df):
    raise RuntimeError(
        "The merge changed the number of evaluations."
    )

print(
    f"Final statistical dataset: {len(correlation_df)} rows"
)

display(
    correlation_df[
        [
            "avaliacao_id",
            "case_id",
            "model",
            "technique",
            "execution",
            "generation_id",
            "nota_final"
        ]
        + detected_metrics
    ].head()
)


Each row of the final dataset corresponds to **one uniquely evaluated BDD**.

Conceptual example:

```text
AV_001
TC_x
model X
few-shot
execution 6
generation_id ...exec-06
nota_final
Manhattan for exec-06
METEOR for exec-06
BERTScore F1 for exec-06
NLI score for exec-06
```

No metric is obtained from row position.


## 15. Descriptive Statistics


In [ ]:

analysis_variables = (
    HUMAN_VARIABLES
    + detected_metrics
)

descriptive_stats = []

for variable in analysis_variables:

    series = pd.to_numeric(
        correlation_df[variable],
        errors="coerce"
    ).dropna()

    if series.empty:
        continue

    descriptive_stats.append({
        "variavel": variable,
        "n": int(len(series)),
        "media": float(series.mean()),
        "mediana": float(series.median()),
        "desvio_padrao": float(series.std(ddof=1)),
        "minimo": float(series.min()),
        "q1": float(series.quantile(0.25)),
        "q3": float(series.quantile(0.75)),
        "maximo": float(series.max())
    })

descriptive_df = pd.DataFrame(
    descriptive_stats
)

display(
    descriptive_df
)


## 16. Shapiro–Wilk — Variable by Variable


In [ ]:

def shapiro_test(series, alpha=ALPHA):

    # Accepts pandas Series, lists, and numpy.ndarray.
    # Conversion to Series guarantees access to dropna()
    # regardless of the received type.
    series = pd.Series(
        series
    )

    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    series = (
        series
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(series) < 3:
        return {
            "n": len(series),
            "W": np.nan,
            "p_value": np.nan,
            "rejeita_normalidade": None
        }

    # Shapiro-Wilk requires some variability.
    # If all values are identical, the distribution
    # cannot be adequately evaluated by the test.
    if series.nunique() < 2:
        return {
            "n": len(series),
            "W": np.nan,
            "p_value": np.nan,
            "rejeita_normalidade": None
        }

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        result = stats.shapiro(
            series.to_numpy(dtype=float)
        )

    return {
        "n": len(series),
        "W": float(result.statistic),
        "p_value": float(result.pvalue),
        "rejeita_normalidade": bool(
            result.pvalue < alpha
        )
    }


normality_results = []

for variable in analysis_variables:

    result = shapiro_test(
        correlation_df[variable]
    )

    normality_results.append({
        "variavel": variable,
        **result
    })


normality_df = pd.DataFrame(
    normality_results
)

display(
    normality_df
)


The test above checks only:

- `nota_final` from the human evaluation;
- each automatic metric.

When a metric contains missing data, the correlation stage also recalculates Shapiro–Wilk on the **exact paired subset** used for that correlation.


## 17. Histograms


In [ ]:

for variable in analysis_variables:

    series = pd.to_numeric(
        correlation_df[variable],
        errors="coerce"
    ).dropna()

    if series.empty:
        continue

    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        series,
        bins="auto"
    )

    plt.title(
        f"Histogram — {variable}"
    )

    plt.xlabel(
        variable
    )

    plt.ylabel(
        "Frequency"
    )

    plt.tight_layout()
    plt.show()


## 18. Q-Q Plots


In [ ]:

for variable in analysis_variables:

    series = pd.to_numeric(
        correlation_df[variable],
        errors="coerce"
    ).dropna()

    if len(series) < 3:
        continue

    plt.figure(
        figsize=(7, 5)
    )

    stats.probplot(
        series,
        dist="norm",
        plot=plt
    )

    plt.title(
        f"Q-Q Plot — {variable}"
    )

    plt.tight_layout()
    plt.show()


## 19. Statistical Functions


In [ ]:

def prepare_pair(df, x_column, y_column):

    temp = df[
        [
            x_column,
            y_column
        ]
    ].copy()

    temp[x_column] = pd.to_numeric(
        temp[x_column],
        errors="coerce"
    )

    temp[y_column] = pd.to_numeric(
        temp[y_column],
        errors="coerce"
    )

    return (
        temp
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )


def outlier_diagnostics(series):

    series = pd.Series(
        series,
        dtype=float
    ).dropna()

    if series.empty:
        return {
            "outliers": 0,
            "outliers_extremos": 0,
            "percentual_outliers": np.nan
        }

    q1 = series.quantile(
        0.25
    )

    q3 = series.quantile(
        0.75
    )

    iqr = q3 - q1

    if iqr == 0:
        return {
            "outliers": 0,
            "outliers_extremos": 0,
            "percentual_outliers": 0.0
        }

    outliers = (
        (series < q1 - 1.5 * iqr)
        | (series > q3 + 1.5 * iqr)
    )

    extreme_outliers = (
        (series < q1 - 3.0 * iqr)
        | (series > q3 + 3.0 * iqr)
    )

    return {
        "outliers": int(outliers.sum()),
        "outliers_extremos": int(extreme_outliers.sum()),
        "percentual_outliers": float(
            100 * outliers.sum() / len(series)
        )
    }


def calculate_r2(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    ss_res = np.sum(
        (y_true - y_pred) ** 2
    )

    ss_tot = np.sum(
        (y_true - np.mean(y_true)) ** 2
    )

    if ss_tot == 0:
        return np.nan

    return (
        1 - ss_res / ss_tot
    )


def linearity_diagnostics(x, y):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    if (
        len(x) < 5
        or len(np.unique(x)) < 3
    ):
        return {
            "r2_linear": np.nan,
            "r2_quadratico": np.nan,
            "delta_r2": np.nan,
            "aproximadamente_linear": None
        }

    try:
        linear = np.polyfit(
            x,
            y,
            deg=1
        )

        y_linear = np.polyval(
            linear,
            x
        )

        r2_linear = calculate_r2(
            y,
            y_linear
        )

        quadratic = np.polyfit(
            x,
            y,
            deg=2
        )

        y_quadratic = np.polyval(
            quadratic,
            x
        )

        r2_quad = calculate_r2(
            y,
            y_quadratic
        )

        delta = (
            r2_quad
            - r2_linear
        )

        return {
            "r2_linear": float(r2_linear),
            "r2_quadratico": float(r2_quad),
            "delta_r2": float(delta),
            "aproximadamente_linear": bool(
                delta <= NONLINEAR_DELTA_R2_THRESHOLD
            )
        }

    except Exception:

        return {
            "r2_linear": np.nan,
            "r2_quadratico": np.nan,
            "delta_r2": np.nan,
            "aproximadamente_linear": None
        }


def safe_correlation(x, y, method):

    if len(x) < 3:
        return np.nan, np.nan

    if (
        np.std(x) == 0
        or np.std(y) == 0
    ):
        return np.nan, np.nan

    if method == "pearson":
        result = stats.pearsonr(
            x,
            y
        )

    elif method == "spearman":
        result = stats.spearmanr(
            x,
            y
        )

    else:
        raise ValueError(
            method
        )

    return (
        float(result.statistic),
        float(result.pvalue)
    )


def bootstrap_correlation(
    x,
    y,
    method,
    n_bootstrap=N_BOOTSTRAP,
    seed=SEED_BOOTSTRAP
):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    n = len(x)

    if n < 4:
        return np.nan, np.nan

    rng = np.random.default_rng(
        seed
    )

    values = []

    for _ in range(n_bootstrap):

        indices = rng.integers(
            0,
            n,
            size=n
        )

        xb = x[
            indices
        ]

        yb = y[
            indices
        ]

        if (
            np.std(xb) == 0
            or np.std(yb) == 0
        ):
            continue

        if method == "pearson":
            value = stats.pearsonr(
                xb,
                yb
            ).statistic

        else:
            value = stats.spearmanr(
                xb,
                yb
            ).statistic

        if np.isfinite(value):
            values.append(
                float(value)
            )

    if len(values) < 100:
        return np.nan, np.nan

    return tuple(
        float(x)
        for x in np.percentile(
            values,
            [
                2.5,
                97.5
            ]
        )
    )


def holm_bonferroni(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    output = np.full(
        len(p_values),
        np.nan
    )

    valid_indices = np.where(
        np.isfinite(p_values)
    )[0]

    if len(valid_indices) == 0:
        return output

    p = p_values[
        valid_indices
    ]

    order = np.argsort(
        p
    )

    ordered_values = p[
        order
    ]

    m = len(ordered_values)

    adjusted_values = np.empty(
        m
    )

    cumulative_value = 0.0

    for i, value in enumerate(ordered_values):

        adjusted_value = min(
            1.0,
            (m - i) * value
        )

        cumulative_value = max(
            cumulative_value,
            adjusted_value
        )

        adjusted_values[i] = cumulative_value

    reverse_values = np.empty(
        m
    )

    reverse_values[
        order
    ] = adjusted_values

    output[
        valid_indices
    ] = reverse_values

    return output


## 20. Correlation and Pair-Specific Decision


In [ ]:

orientation_map = {
    name: config["orientacao"]
    for name, config in METRICS_CONFIG.items()
}


results = []

for human_variable in HUMAN_VARIABLES:

    for metric in detected_metrics:

        pair = prepare_pair(
            correlation_df,
            human_variable,
            metric
        )

        if len(pair) < 3:
            continue

        x = pair[
            human_variable
        ].to_numpy(
            dtype=float
        )

        y = pair[
            metric
        ].to_numpy(
            dtype=float
        )

        # ----------------------------------------------------
        # NORMALITY IN THE EXACT PAIRED SUBSET
        # ----------------------------------------------------

        human_shapiro = shapiro_test(
            x
        )

        metric_shapiro = shapiro_test(
            y
        )

        both_normal = (
            human_shapiro["p_value"] >= ALPHA
            and metric_shapiro["p_value"] >= ALPHA
        )

        # ----------------------------------------------------
        # OUTLIERS
        # ----------------------------------------------------

        human_outlier_info = outlier_diagnostics(
            x
        )

        metric_outlier_info = outlier_diagnostics(
            y
        )

        total_extreme_outliers = (
            human_outlier_info["outliers_extremos"]
            + metric_outlier_info["outliers_extremos"]
        )

        # ----------------------------------------------------
        # LINEARITY
        # ----------------------------------------------------

        linearity = linearity_diagnostics(
            x,
            y
        )

        # ----------------------------------------------------
        # CORRELATIONS
        # ----------------------------------------------------

        pearson_r, pearson_p = safe_correlation(
            x,
            y,
            "pearson"
        )

        spearman_rho, spearman_p = safe_correlation(
            x,
            y,
            "spearman"
        )

        pearson_lower, pearson_upper = bootstrap_correlation(
            x,
            y,
            "pearson"
        )

        spearman_lower, spearman_upper = bootstrap_correlation(
            x,
            y,
            "spearman"
        )

        # ----------------------------------------------------
        # RECOMMENDATION
        # ----------------------------------------------------

        if (
            both_normal
            and total_extreme_outliers == 0
            and linearity["aproximadamente_linear"] is True
        ):

            method = "Pearson"

            reason = (
                "both variables do not reject normality; "
                "no extreme outliers; "
                "approximately linear relationship"
            )

        else:

            method = "Spearman"

            reasons = []

            if not both_normal:
                reasons.append(
                    "normality not satisfied by at least one variable"
                )

            if total_extreme_outliers > 0:
                reasons.append(
                    "presence of extreme outliers"
                )

            if linearity["aproximadamente_linear"] is False:
                reasons.append(
                    "indication of nonlinearity"
                )

            if linearity["aproximadamente_linear"] is None:
                reasons.append(
                    "linearity not automatically determined"
                )

            reason = "; ".join(
                reasons
            )

        orientation = orientation_map.get(
            metric,
            +1
        )

        results.append({
            "variavel_humana": human_variable,
            "metrica": metric,
            "n": len(pair),

            "shapiro_humana_W": human_shapiro["W"],
            "shapiro_humana_p": human_shapiro["p_value"],

            "shapiro_metrica_W": metric_shapiro["W"],
            "shapiro_metrica_p": metric_shapiro["p_value"],

            "ambas_nao_rejeitam_normalidade": both_normal,

            "outliers_humana": human_outlier_info["outliers"],
            "outliers_metrica": metric_outlier_info["outliers"],
            "outliers_extremos_total": total_extreme_outliers,

            "r2_linear": linearity["r2_linear"],
            "r2_quadratico": linearity["r2_quadratico"],
            "delta_r2": linearity["delta_r2"],
            "aproximadamente_linear": linearity["aproximadamente_linear"],

            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "pearson_ic95_inf": pearson_lower,
            "pearson_ic95_sup": pearson_upper,

            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
            "spearman_ic95_inf": spearman_lower,
            "spearman_ic95_sup": spearman_upper,

            "orientacao_metrica": orientation,

            "pearson_orientado": (
                pearson_r * orientation
                if np.isfinite(pearson_r)
                else np.nan
            ),

            "spearman_orientado": (
                spearman_rho * orientation
                if np.isfinite(spearman_rho)
                else np.nan
            ),

            "metodo_recomendado": method,
            "motivo_recomendacao": reason
        })


correlations_df = pd.DataFrame(
    results
)

display(
    correlations_df
)


## 21. Holm Correction


In [ ]:

if not correlations_df.empty:

    correlations_df[
        "pearson_p_holm"
    ] = holm_bonferroni(
        correlations_df[
            "pearson_p"
        ].to_numpy()
    )

    correlations_df[
        "spearman_p_holm"
    ] = holm_bonferroni(
        correlations_df[
            "spearman_p"
        ].to_numpy()
    )


display(
    correlations_df[
        [
            "variavel_humana",
            "metrica",
            "n",
            "pearson_r",
            "pearson_p",
            "pearson_p_holm",
            "spearman_rho",
            "spearman_p",
            "spearman_p_holm",
            "metodo_recomendado"
        ]
    ]
)


## 22. Main Table — Final Score × Metrics


In [ ]:

final_score_correlation_df = (
    correlations_df[
        correlations_df[
            "variavel_humana"
        ] == "nota_final"
    ]
    .copy()
    .sort_values(
        "spearman_orientado",
        ascending=False
    )
)


display(
    final_score_correlation_df[
        [
            "metrica",
            "n",

            "shapiro_humana_p",
            "shapiro_metrica_p",

            "outliers_extremos_total",
            "aproximadamente_linear",

            "pearson_r",
            "pearson_ic95_inf",
            "pearson_ic95_sup",
            "pearson_p_holm",

            "spearman_rho",
            "spearman_ic95_inf",
            "spearman_ic95_sup",
            "spearman_p_holm",

            "pearson_orientado",
            "spearman_orientado",

            "metodo_recomendado",
            "motivo_recomendacao"
        ]
    ]
)


## 23. Spearman Summary — Final Score × Metrics


In [ ]:
spearman_table = (
    correlations_df[
        correlations_df["variavel_humana"] == "nota_final"
    ][
        [
            "metrica",
            "n",
            "spearman_rho",
            "spearman_orientado",
            "spearman_p",
            "spearman_p_holm",
            "metodo_recomendado"
        ]
    ]
    .copy()
    .sort_values(
        "spearman_orientado",
        ascending=False
    )
)

display(
    spearman_table
)


## 24. Scatter Plots


In [ ]:

plot_criteria = (
    HUMAN_VARIABLES
    if GENERATE_PLOTS_FOR_ALL_CRITERIA
    else ["nota_final"]
)

for human_variable in plot_criteria:

    for metric in detected_metrics:

        pair = prepare_pair(
            correlation_df,
            human_variable,
            metric
        )

        if len(pair) < 3:
            continue

        x = pair[
            metric
        ].to_numpy(
            dtype=float
        )

        y = pair[
            human_variable
        ].to_numpy(
            dtype=float
        )

        plt.figure(
            figsize=(8, 5)
        )

        plt.scatter(
            x,
            y,
            alpha=0.7
        )

        if (
            len(np.unique(x)) >= 2
            and np.std(x) > 0
        ):

            coefficients = np.polyfit(
                x,
                y,
                deg=1
            )

            order = np.argsort(
                x
            )

            plt.plot(
                x[order],
                np.polyval(
                    coefficients,
                    x[order]
                )
            )

        plt.xlabel(
            metric
        )

        plt.ylabel(
            human_variable
        )

        plt.title(
            f"{human_variable} × {metric}"
        )

        plt.tight_layout()
        plt.show()


## 25. Sensitivity Analysis — Complete Cases


In [ ]:

complete_cases_df = (
    correlation_df
    .dropna(
        subset=detected_metrics
    )
    .copy()
)

print(
    f"Complete cases across all metrics: "
    f"{len(complete_cases_df)} / {len(correlation_df)}"
)


complete_results = []

for human_variable in HUMAN_VARIABLES:

    for metric in detected_metrics:

        pair = prepare_pair(
            complete_cases_df,
            human_variable,
            metric
        )

        if len(pair) < 3:
            continue

        x = pair[
            human_variable
        ].to_numpy(
            dtype=float
        )

        y = pair[
            metric
        ].to_numpy(
            dtype=float
        )

        pearson_r, pearson_p = safe_correlation(
            x,
            y,
            "pearson"
        )

        spearman_rho, spearman_p = safe_correlation(
            x,
            y,
            "spearman"
        )

        orientation = orientation_map.get(
            metric,
            +1
        )

        complete_results.append({
            "variavel_humana": human_variable,
            "metrica": metric,
            "n": len(pair),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
            "pearson_orientado": (
                pearson_r * orientation
                if np.isfinite(pearson_r)
                else np.nan
            ),
            "spearman_orientado": (
                spearman_rho * orientation
                if np.isfinite(spearman_rho)
                else np.nan
            )
        })


complete_correlations_df = pd.DataFrame(
    complete_results
)

if not complete_correlations_df.empty:

    complete_correlations_df[
        "pearson_p_holm"
    ] = holm_bonferroni(
        complete_correlations_df[
            "pearson_p"
        ].to_numpy()
    )

    complete_correlations_df[
        "spearman_p_holm"
    ] = holm_bonferroni(
        complete_correlations_df[
            "spearman_p"
        ].to_numpy()
    )


display(
    complete_correlations_df
)


## 26. Direct Statistical Comparison Between Metrics

Ranking metrics by the observed Spearman coefficient shows **which metric has the highest observed association** with `nota_final`.

However, a correlation of `0.38` is not automatically statistically superior to a correlation of `0.24`.

Therefore, this stage directly compares metrics in pairs using **paired bootstrap**.

For each pair of metrics:

1. only BDDs containing `nota_final` and both metrics are selected;
2. the same rows are resampled for both metrics;
3. the oriented Spearman coefficient is calculated for each metric;
4. the following is calculated:

\[
\Delta \rho =
\rho_{\text{metric A, oriented}}
-
\rho_{\text{metric B, oriented}}
\]

5. the process is repeated `N_BOOTSTRAP_COMPARISON` times;
6. the 95% CI of the difference is obtained;
7. a two-sided bootstrap p-value is calculated;
8. Holm correction is applied to the multiple comparisons.

### Interpretation

If the 95% CI of \(\Delta\rho\) **does not include zero** and `p_holm < 0.05`, there is evidence of a statistically supported difference between the correlations.

If the interval includes zero, the metric with the larger coefficient still has the **highest observed correlation**, but the difference between the two metrics is considered inconclusive.


In [ ]:

from itertools import combinations


def safe_oriented_spearman(score, metric, orientation):
    score = np.asarray(score, dtype=float)
    metric = np.asarray(metric, dtype=float)

    if len(score) < 3:
        return np.nan

    if np.std(score) == 0 or np.std(metric) == 0:
        return np.nan

    rho = stats.spearmanr(
        score,
        metric
    ).statistic

    if not np.isfinite(rho):
        return np.nan

    return float(
        rho * orientation
    )


def compare_two_metrics_bootstrap(
    df,
    metric_a,
    metric_b,
    orientation_a,
    orientation_b,
    n_bootstrap=N_BOOTSTRAP_COMPARISON,
    seed=SEED_BOOTSTRAP
):
    # Same subset for both metrics
    temp = df[
        [
            "nota_final",
            metric_a,
            metric_b
        ]
    ].copy()

    for column in [
        "nota_final",
        metric_a,
        metric_b
    ]:
        temp[column] = pd.to_numeric(
            temp[column],
            errors="coerce"
        )

    temp = (
        temp
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    n = len(temp)

    if n < 4:
        return {
            "n": n,
            "rho_a_orientado": np.nan,
            "rho_b_orientado": np.nan,
            "delta_observado": np.nan,
            "ic95_inf": np.nan,
            "ic95_sup": np.nan,
            "p_bootstrap": np.nan,
            "n_bootstrap_validos": 0
        }

    score = temp[
        "nota_final"
    ].to_numpy(
        dtype=float
    )

    a = temp[
        metric_a
    ].to_numpy(
        dtype=float
    )

    b = temp[
        metric_b
    ].to_numpy(
        dtype=float
    )

    rho_a = safe_oriented_spearman(
        score,
        a,
        orientation_a
    )

    rho_b = safe_oriented_spearman(
        score,
        b,
        orientation_b
    )

    observed_delta = (
        rho_a - rho_b
        if np.isfinite(rho_a)
        and np.isfinite(rho_b)
        else np.nan
    )

    rng = np.random.default_rng(
        seed
    )

    deltas = []

    for _ in range(n_bootstrap):
        indices = rng.integers(
            0,
            n,
            size=n
        )

        score_b = score[
            indices
        ]

        a_b = a[
            indices
        ]

        b_b = b[
            indices
        ]

        rho_a_b = safe_oriented_spearman(
            score_b,
            a_b,
            orientation_a
        )

        rho_b_b = safe_oriented_spearman(
            score_b,
            b_b,
            orientation_b
        )

        if (
            np.isfinite(rho_a_b)
            and np.isfinite(rho_b_b)
        ):
            deltas.append(
                rho_a_b - rho_b_b
            )

    deltas = np.asarray(
        deltas,
        dtype=float
    )

    if len(deltas) < 100:
        return {
            "n": n,
            "rho_a_orientado": rho_a,
            "rho_b_orientado": rho_b,
            "delta_observado": observed_delta,
            "ic95_inf": np.nan,
            "ic95_sup": np.nan,
            "p_bootstrap": np.nan,
            "n_bootstrap_validos": len(deltas)
        }

    ic95_inf, ic95_sup = np.percentile(
        deltas,
        [
            2.5,
            97.5
        ]
    )

    # Two-sided bootstrap p-value around zero
    proportion_le_zero = np.mean(
        deltas <= 0
    )

    proportion_ge_zero = np.mean(
        deltas >= 0
    )

    p_bootstrap = min(
        1.0,
        2.0
        * min(
            proportion_le_zero,
            proportion_ge_zero
        )
    )

    return {
        "n": n,
        "rho_a_orientado": rho_a,
        "rho_b_orientado": rho_b,
        "delta_observado": float(
            observed_delta
        ),
        "ic95_inf": float(
            ic95_inf
        ),
        "ic95_sup": float(
            ic95_sup
        ),
        "p_bootstrap": float(
            p_bootstrap
        ),
        "n_bootstrap_validos": int(
            len(deltas)
        )
    }


# ============================================================
# ONLY THE MAIN METRICS ENTER THE
# "BEST METRIC" COMPARISON
# ============================================================

ranking_metrics = [
    metric
    for metric in MAIN_METRICS
    if metric in correlation_df.columns
]

if len(ranking_metrics) < 2:
    raise RuntimeError(
        "At least two main metrics are required "
        "to perform the direct statistical comparison."
    )


metric_comparisons = []

for metric_a, metric_b in combinations(
    ranking_metrics,
    2
):
    result = compare_two_metrics_bootstrap(
        df=correlation_df,
        metric_a=metric_a,
        metric_b=metric_b,
        orientation_a=orientation_map.get(
            metric_a,
            +1
        ),
        orientation_b=orientation_map.get(
            metric_b,
            +1
        )
    )

    metric_comparisons.append({
        "metrica_a": metric_a,
        "metrica_b": metric_b,
        **result
    })


metric_comparisons_df = pd.DataFrame(
    metric_comparisons
)

if not metric_comparisons_df.empty:
    metric_comparisons_df[
        "p_holm"
    ] = holm_bonferroni(
        metric_comparisons_df[
            "p_bootstrap"
        ].to_numpy()
    )

    metric_comparisons_df[
        "ic95_exclui_zero"
    ] = (
        (
            metric_comparisons_df[
                "ic95_inf"
            ] > 0
        )
        |
        (
            metric_comparisons_df[
                "ic95_sup"
            ] < 0
        )
    )

    metric_comparisons_df[
        "diferenca_significativa"
    ] = (
        metric_comparisons_df[
            "ic95_exclui_zero"
        ]
        &
        (
            metric_comparisons_df[
                "p_holm"
            ] < ALPHA
        )
    )

    def interpret_comparison(row):
        if not row[
            "diferenca_significativa"
        ]:
            return (
                "inconclusive difference"
            )

        if row[
            "delta_observado"
        ] > 0:
            return (
                f"{row['metrica_a']} > "
                f"{row['metrica_b']}"
            )

        if row[
            "delta_observado"
        ] < 0:
            return (
                f"{row['metrica_b']} > "
                f"{row['metrica_a']}"
            )

        return (
            "no observed difference"
        )

    metric_comparisons_df[
        "interpretacao"
    ] = metric_comparisons_df.apply(
        interpret_comparison,
        axis=1
    )


display(
    metric_comparisons_df[
        [
            "metrica_a",
            "metrica_b",
            "n",
            "rho_a_orientado",
            "rho_b_orientado",
            "delta_observado",
            "ic95_inf",
            "ic95_sup",
            "p_bootstrap",
            "p_holm",
            "diferenca_significativa",
            "interpretacao"
        ]
    ]
)


## 27. Metric Ranking

The main ranking uses **oriented Spearman** with `nota_final`.

The first-ranked metric is the one with the highest observed association.

The bootstrap comparison from the previous section indicates whether this advantage is also statistically supported.


In [ ]:

# ============================================================
# OBSERVED RANKING
# ============================================================

metric_ranking_df = (
    final_score_correlation_df[
        final_score_correlation_df[
            "metrica"
        ].isin(
            ranking_metrics
        )
    ][
        [
            "metrica",
            "n",
            "spearman_rho",
            "spearman_orientado",
            "spearman_ic95_inf",
            "spearman_ic95_sup",
            "spearman_p",
            "spearman_p_holm",
            "pearson_r",
            "pearson_orientado",
            "metodo_recomendado"
        ]
    ]
    .copy()
    .sort_values(
        "spearman_orientado",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

metric_ranking_df.insert(
    0,
    "posicao",
    np.arange(
        1,
        len(metric_ranking_df) + 1
    )
)


display(
    metric_ranking_df
)


# ============================================================
# INTERPRETATION OF THE FIRST-RANKED METRIC
# ============================================================

best_observed_metric = (
    metric_ranking_df.iloc[0][
        "metrica"
    ]
)

best_observed_rho = float(
    metric_ranking_df.iloc[0][
        "spearman_orientado"
    ]
)

print(
    "=" * 80
)

print(
    "METRIC RANKING — CORRELATION WITH THE HUMAN FINAL SCORE"
)

print(
    "=" * 80
)

for _, row in metric_ranking_df.iterrows():
    print(
        f"{int(row['posicao'])}º "
        f"{row['metrica']}: "
        f"Oriented Spearman = "
        f"{row['spearman_orientado']:.4f}"
    )

print(
    "\\nMETRIC WITH THE HIGHEST OBSERVED ASSOCIATION:"
)

print(
    f"{best_observed_metric} "
    f"(oriented rho = {best_observed_rho:.4f})"
)


# ============================================================
# IS THE FIRST-RANKED METRIC STATISTICALLY SUPERIOR
# TO THE OTHERS?
# ============================================================

best_metric_comparisons = metric_comparisons_df[
    (
        metric_comparisons_df[
            "metrica_a"
        ] == best_observed_metric
    )
    |
    (
        metric_comparisons_df[
            "metrica_b"
        ] == best_observed_metric
    )
].copy()

superior_to_all = True
best_metric_results = []

for _, row in best_metric_comparisons.iterrows():

    other_metric = (
        row["metrica_b"]
        if row["metrica_a"]
        == best_observed_metric
        else row["metrica_a"]
    )

    if (
        row["metrica_a"]
        == best_observed_metric
    ):
        best_metric_delta = (
            row["delta_observado"]
        )
    else:
        best_metric_delta = (
            -row["delta_observado"]
        )

    statistically_superior = bool(
        row[
            "diferenca_significativa"
        ]
        and best_metric_delta > 0
    )

    best_metric_results.append({
        "comparada_com": other_metric,
        "superior_estatisticamente": statistically_superior,
        "p_holm": row["p_holm"],
        "interpretacao_original": row[
            "interpretacao"
        ]
    })

    if not statistically_superior:
        superior_to_all = False


best_metric_superiority_df = pd.DataFrame(
    best_metric_results
)

print(
    "\\nCOMPARISON OF THE FIRST-RANKED METRIC WITH THE OTHERS:"
)

display(
    best_metric_superiority_df
)


if superior_to_all:
    best_metric_conclusion = (
        f"{best_observed_metric} showed the highest "
        "observed correlation and was statistically superior "
        "to all other metrics in the paired bootstrap "
        "comparisons with Holm correction."
    )
else:
    best_metric_conclusion = (
        f"{best_observed_metric} showed the highest "
        "observed correlation with the human final score, "
        "but statistical superiority over all "
        "other metrics was not demonstrated."
    )

print(
    "\\nCONCLUSION:"
)

print(
    best_metric_conclusion
)


## 28. Export All Results


In [ ]:

# Inventory of the 60 files
inventory_df.to_csv(
    OUTPUT_FOLDER / "inventario_arquivos_metricas.csv",
    index=False,
    encoding="utf-8-sig"
)

# Coverage by file after the human-evaluation filter
file_coverage_df.to_csv(
    OUTPUT_FOLDER / "cobertura_por_arquivo.csv",
    index=False,
    encoding="utf-8-sig"
)

# Statistical dataset
correlation_df.to_csv(
    OUTPUT_FOLDER / "dataset_correlacao.csv",
    index=False,
    encoding="utf-8-sig"
)

correlation_df.to_json(
    OUTPUT_FOLDER / "dataset_correlacao.json",
    orient="records",
    force_ascii=False,
    indent=2
)

# Coverage
coverage_df.to_csv(
    OUTPUT_FOLDER / "cobertura_metricas.csv",
    index=False,
    encoding="utf-8-sig"
)

# Descriptive statistics
descriptive_df.to_csv(
    OUTPUT_FOLDER / "estatistica_descritiva.csv",
    index=False,
    encoding="utf-8-sig"
)

# Normality
normality_df.to_csv(
    OUTPUT_FOLDER / "normalidade_shapiro.csv",
    index=False,
    encoding="utf-8-sig"
)

# Correlations
correlations_df.to_csv(
    OUTPUT_FOLDER / "correlacoes_pares_disponiveis.csv",
    index=False,
    encoding="utf-8-sig"
)

final_score_correlation_df.to_csv(
    OUTPUT_FOLDER / "correlacoes_nota_final.csv",
    index=False,
    encoding="utf-8-sig"
)

# Complete cases
complete_correlations_df.to_csv(
    OUTPUT_FOLDER / "correlacoes_casos_completos.csv",
    index=False,
    encoding="utf-8-sig"
)

# Spearman summary — final score × metrics
spearman_table.to_csv(
    OUTPUT_FOLDER / "spearman_nota_final.csv",
    index=False,
    encoding="utf-8-sig"
)




# Direct comparisons between metrics
metric_comparisons_df.to_csv(
    OUTPUT_FOLDER / "comparacoes_pareadas_metricas.csv",
    index=False,
    encoding="utf-8-sig"
)

# Metric ranking
metric_ranking_df.to_csv(
    OUTPUT_FOLDER / "ranking_metricas.csv",
    index=False,
    encoding="utf-8-sig"
)

# Comparison of the best observed metric with the others
best_metric_superiority_df.to_csv(
    OUTPUT_FOLDER / "superioridade_melhor_metrica.csv",
    index=False,
    encoding="utf-8-sig"
)

# Automatic textual conclusion
with open(
    OUTPUT_FOLDER / "conclusao_melhor_metrica.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(
        best_metric_conclusion
        + "\\n"
    )


print(
    "Files generated in:"
)

print(
    OUTPUT_FOLDER.resolve()
)

for file in sorted(
    OUTPUT_FOLDER.glob("*")
):
    print(
        " -",
        file.name
    )


## 29. Automatic Summary


In [ ]:

print(
    "=" * 80
)

print(
    "SUMMARY"
)

print(
    "=" * 80
)

print(
    f"\nMetric files: {len(inventory_df)}"
)

print(
    f"Models: {len(found_models)}"
)

print(
    f"Techniques: {len(found_techniques)}"
)

print(
    f"Families: {len(found_families)}"
)

print(
    f"Human evaluations: {len(correlation_df)}"
)

print(
    "\nMetrics analyzed:"
)

for metric in detected_metrics:
    print(
        " -",
        metric
    )


print(
    "\n" + "=" * 80
)

print(
    "FINAL SCORE × METRICS"
)

print(
    "=" * 80
)


for _, row in final_score_correlation_df.iterrows():

    print(
        f"\n{row['metrica']}"
    )

    print(
        f"  N = {int(row['n'])}"
    )

    print(
        f"  Pearson r = {row['pearson_r']:.4f}"
    )

    print(
        f"  Spearman rho = {row['spearman_rho']:.4f}"
    )

    print(
        f"  Oriented Spearman = {row['spearman_orientado']:.4f}"
    )

    print(
        f"  Recommended method = {row['metodo_recomendado']}"
    )

    print(
        f"  Reason = {row['motivo_recomendacao']}"
    )


print(
    "\n" + "=" * 80
)

print(
    "FINAL RANKING — ORIENTED SPEARMAN"
)

print(
    "=" * 80
)

for _, row in metric_ranking_df.iterrows():
    print(
        f"{int(row['posicao'])}º "
        f"{row['metrica']}: "
        f"{row['spearman_orientado']:.4f}"
    )

print(
    "\nCONCLUSION ABOUT THE BEST METRIC:"
)

print(
    best_metric_conclusion
)


In [ ]:
# ============================================================
# BERTSCORE COMPARISONS WITH THE OTHER METRICS
# Table for the paper
# ============================================================

BERT = "bertscore_f1"

bert_comparisons = []

for _, row in metric_comparisons_df.iterrows():

    if row["metrica_a"] == BERT:

        other_metric = row["metrica_b"]

        bert_rho = row["rho_a_orientado"]
        other_rho = row["rho_b_orientado"]

        delta = row["delta_observado"]
        ci_low = row["ic95_inf"]
        ci_high = row["ic95_sup"]

    elif row["metrica_b"] == BERT:

        other_metric = row["metrica_a"]

        bert_rho = row["rho_b_orientado"]
        other_rho = row["rho_a_orientado"]

        # Reverse the direction to always keep:
        # delta = rho_BERTScore - rho_other
        delta = -row["delta_observado"]

        # When reversing the sign, the limits are also reversed
        ci_low = -row["ic95_sup"]
        ci_high = -row["ic95_inf"]

    else:
        continue

    bert_comparisons.append({
        "comparacao": f"BERTScore vs. {other_metric}",
        "n": int(row["n"]),
        "rho_bertscore": bert_rho,
        "rho_comparada": other_rho,
        "delta_rho": delta,
        "ci95_inf": ci_low,
        "ci95_sup": ci_high,
        "p_holm": row["p_holm"],
        "significativa": row["diferenca_significativa"]
    })


bert_table = pd.DataFrame(bert_comparisons)

print("\nBERTSCORE COMPARISONS WITH THE OTHER METRICS\n")

display(
    bert_table.style.format({
        "rho_bertscore": "{:.4f}",
        "rho_comparada": "{:.4f}",
        "delta_rho": "{:.4f}",
        "ci95_inf": "{:.4f}",
        "ci95_sup": "{:.4f}",
        "p_holm": "{:.6g}"
    })
)

bert_table.to_csv(
    OUTPUT_FOLDER / "comparacoes_bertscore_paper.csv",
    index=False,
    encoding="utf-8-sig"
)

# Statistical Strategy

## Main analysis

The methodological recommendation is to state in advance:

> **Spearman as the primary analysis**, with Pearson presented as a complementary analysis.

For each pair, the notebook still checks:

- Shapiro–Wilk for the human variable in the paired subset;
- Shapiro–Wilk for the metric;
- outliers;
- linearity.

When all evaluated assumptions are adequate, the notebook marks Pearson as appropriate.

## Distance metrics

Manhattan has orientation `-1`.

Thus:

```text
raw rho = -0.70
oriented rho = +0.70
```

does not change the original statistical result; it only facilitates directional comparison with metrics in which higher values represent better quality.

## NLI metrics

The primary analysis uses:

```text
nli_score
```

The components:

```text
entailment_mean
neutral_mean
contradiction_mean
contradiction_rate
coverage
```

are treated separately and are included only if:

```python
ANALYZE_SECONDARY_COMPONENTS = True
```

This prevents the aggregated NLI score from being confused with only the mean entailment probability and also avoids unnecessarily increasing the number of tests in the primary analysis.

## Comparison Between Metrics

The term **"best metric"** is handled at two levels:

1. **highest observed correlation**: highest `oriented Spearman`;
2. **statistical superiority**: paired Spearman difference evaluated by bootstrap, with 95% CI and Holm correction.

Thus, the notebook avoids claiming that one metric is statistically superior merely because its observed coefficient is numerically larger.
